# COMP5122M – Data Science Coursework (Part 1)

## Project: Vibe Check - How Gen Z Really Feels About AI
## Team - Shyamin

**Members:** Durgam Naga Deeraj Reddy (pvcl0430) ; Harshini Gonela (jklt0901) ; Karuna Anil Gholap (lctl0641) ; Rehan Ali (fnkx0448)

## Task 1: Data Quality and Preprocessing
**Performed by:** Durgam Naga Deeraj Reddy (pvcl0430)

### 1.1 Load Raw Dataset and Normalise Missing Strings
Loads the Pew Global Attitudes Spring 2025 CSV file and converts blank or whitespace-only cells into standard `NaN` values.

In [21]:
import pandas as pd # import pandas for dataframe operations
import numpy as np # import numpy for missing value handling
import re # import regex for pattern matching

raw_path = "/Users/sdreddy/Leeds/Sem 1/Data Science/coursework 1/Resources/Pew_Original_Data/Pew Research Center Global Attitudes Spring 2025 Dataset CSV.csv" # define raw dataset path
df = pd.read_csv(raw_path, low_memory=False) # load raw survey dataset
df = df.replace(r"^\s*$", np.nan, regex=True).infer_objects(copy=False) # replace whitespace strings with nan

/var/folders/15/vnzgb5095kj14qn7m2szdjr80000gn/T/ipykernel_58524/4023692406.py:7: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  df = df.replace(r"^\s*$", np.nan, regex=True).infer_objects(copy=False) # replace whitespace strings with nan


### 1.2 Parse Official Data Dictionary for Non-Response Codes
Extracts variable-specific numeric codes representing `Don't know` and `Refused` responses from the official Excel dictionary.

In [22]:
dict_path = "../Resources/Pew_Original_Data/Pew Research Center Global Attitudes Spring 2025 Data Dictionary.xlsx" # define dictionary path
values_dictionary = pd.read_excel(dict_path, sheet_name="Values", header=None).iloc[2:].copy() # read value labels sheet
values_dictionary.columns = ["variable", "value_code", "value_label"] # assign column names
values_dictionary["variable"] = values_dictionary["variable"].ffill() # forward-fill merged variable names
values_dictionary = values_dictionary.dropna(subset=["variable", "value_code", "value_label"]).copy() # drop incomplete dictionary rows

values_dictionary["value_label"] = ( # standardise apostrophes and whitespace in labels
    values_dictionary["value_label"].astype(str)
    .str.replace("’", "'", regex=False)
    .str.replace("‘", "'", regex=False)
    .str.strip()
)

special_codes = values_dictionary[ # filter don't know and refused labels
    values_dictionary["value_label"].str.contains(r"\bdon't\s+know\b|\brefused\b", case=False, regex=True, na=False)
].copy()
special_codes["value_code"] = pd.to_numeric(special_codes["value_code"], errors="coerce") # convert codes to numeric
special_codes = special_codes.dropna(subset=["variable", "value_code"]).drop_duplicates(subset=["variable", "value_code"]) # keep unique variable-code pairs

### 1.3 Recode Non-Response Values to Missing (`NaN`)
Replaces identified `Don't know` and `Refused` numeric codes with `NaN` across all matching variables in the dataset.

In [23]:
for _, row in special_codes.iterrows(): # iterate over non-response code rules
    variable, code_value = row["variable"], row["value_code"] # extract variable name and target code
    if variable in df.columns: # verify variable exists in dataset
        df[variable] = pd.to_numeric(df[variable], errors="coerce") # ensure column is numeric
        df.loc[df[variable].eq(code_value), variable] = np.nan # replace special code with nan

### 1.4 Define Cross-National Education Harmonisation Mapping
Maps country-specific education labels into a standardised international classification scheme using exact string matches and keyword rules.

In [24]:
excluded_education_variables = ["d_educ_germany_vocational", "d_educ_israel_years", "d_educ_israel_vocational"] # exclude supplemental education fields
education_variables = [col for col in df.columns if col.startswith("d_educ_") and col not in excluded_education_variables] # select primary country education columns

education_dictionary = values_dictionary[values_dictionary["variable"].isin(education_variables)].copy() # subset dictionary to education variables
education_dictionary["value_code"] = pd.to_numeric(education_dictionary["value_code"], errors="coerce") # convert education codes to numeric

def education_category(label): # map raw education label to harmonised tier
    text = re.sub(r"\s+", " ", str(label).strip().lower().replace("’", "'").replace("‘", "'").replace("–", "-").replace("—", "-")) # normalise label text
    if re.search(r"\bdon't know\b|\brefused\b", text): # return nan for non-responses
        return np.nan

    exact_mappings = { # define exact string overrides for ambiguous labels
        "secondary school or less": "Secondary education (broad category)",
        "bachelor and above": "Tertiary education (broad category)",
        "secondary school - long track, technical college, university entrance": "Upper secondary",
        "secondary school - short track, apprenticeship": "Vocational / technical",
        "complete preparatory": "Upper secondary",
        "senior general secondary (havo/vwo - second three grades)/hbs": "Upper secondary",
        "undergraduate education at the second level - college (brevet des collèges)": "Lower secondary",
        "gcse /cse/gce o-level (any grades) ... [see demographic qq for full wording]": "Lower secondary",
        "undergraduate qualifying program/university transfer program (cegep in quebec)": "Some tertiary / incomplete",
        "upper primary or middle school (upper primary certificate, up to 8th standard)": "Lower secondary",
        "first degree - b.a.": "Undergraduate",
        "honours degree": "Undergraduate",
        "undergraduate teaching university (diploma of general university studies (deug), license, license (lmd))": "Undergraduate",
        "technological educational institute (a.tei) (highest tertiary education), aspaite (after the anotatopoiisi)": "Undergraduate",
        "ptychio (upper tertiary certificate) ... [see demographic qq for full wording]": "Undergraduate",
        "diploma programmes (div) or program sarjana s1 (bachelors degree)": "Undergraduate",
        "tertiary complete, no university": "Vocational / technical",
        "complete post-graduation courses (specialization, professional masters, mba)": "Postgraduate",
        "program spesialis 1 (graduate diploma (si)) and profesi doktor (medical doctor diploma)": "Postgraduate",
        "post graduate diploma": "Postgraduate",
        "second degree - m.a.": "Postgraduate",
        "diploma programmes (di thru diii)": "Vocational / technical",
        "epagelmatiko lykeio (epal), epagelmatiki sxoli ... [see demographic qq for full wording]": "Vocational / technical",
        "second degree second stage ... [see demographic qq for full wording]": "Upper secondary"
    }
    if text in exact_mappings: # check exact dictionary match first
        return exact_mappings[text]
    if re.search(r"\b(no formal|no schooling|no qualifications|no educational qualifications|did not receive formal education|has not attended school|illiterate|pre-primary|preschool|kindergarten|primary|elementary|less than 5 years of schooling|childhood education|early childhood|grades 1-6|1st-7th grades)\b", text): # classify primary or no schooling
        return "No formal / primary"
    if re.search(r"\b(or vocational|some senior high school or vocational)\b", text): # flag mixed secondary/vocational labels
        return "Other - review required"
    if re.search(r"\bincomplete secondary\b", text): # classify broad incomplete secondary
        return "Secondary education (broad category)"
    if re.search(r"\b(lower secondary|junior high|middle school|junior secondary|second grade junior|incomplete junior high|incomplete middle school|grades 7-9|grades 7-8|first three grades|brevet des collèges)\b", text): # classify lower secondary
        return "Lower secondary"
    if re.search(r"\b(doctorate|doctoral|doctor's degree|doctors degree|phd|doctor degree|program doctor|master|postgraduate|post graduate|graduate degree|graduate education|licentiate)\b", text): # classify postgraduate degrees
        return "Postgraduate"
    if re.search(r"\b(incomplete tertiary|incomplete university|incomplete higher education|incomplete higher education course|university transfer program)\b", text): # classify incomplete tertiary
        return "Some tertiary / incomplete"
    if re.search(r"\b(bachelor|b\.a\.|bsc|b\.sc|b\.tech|complete university|university degree|university graduates|university/college graduates|three-year degree|college degree|college bachelor|first degree|university diploma|deug|licence \(lmd\)|associate's degree|associate degree|undergraduate)\b", text): # classify undergraduate degrees
        return "Undergraduate"
    if re.search(r"\b(vocational|technical|trade|apprenticeship|apprentice|polytechnic|professional training|professional specialization|specialist training|advanced vocational|higher vocational|training institute|teacher training|craft trade|industrial training|tivet|nated|mbo|hnc|hnd|nvq|nce|technician|technologist|technical career|commercial career|college of technology|special training|training cycles|vocational school|vocational training|certificate/upgrading|trade certificate|secondary technical|secondary vocational|higher national diploma)\b", text): # classify vocational and technical tracks
        return "Vocational / technical"
    if re.search(r"\b(upper secondary|high school|senior secondary|secondary school|secondary education|complete secondary|completed secondary|a-level|a level|matriculation|baccalaureate|bachillerato|general lykeio|school graduate|havo|vwo|school leaving certificate|certificate of secondary education)\b", text): # classify upper secondary
        return "Upper secondary"
    return "Other - review required" # fallback for unmatched labels

education_dictionary["education_harmonised"] = education_dictionary["value_label"].apply(education_category) # apply harmonisation to dictionary

### 1.5 Apply Harmonised Education and Export Cleaned Dataset
Consolidates respondent education across all country columns into `education_harmonised` and exports `Task1_Cleaned_Dataset.csv` for downstream tasks.

In [25]:
df = df.copy() # defragment dataframe before adding columns
education_source = df[education_variables] # slice country education columns
df["education_source_variable"] = education_source.notna().idxmax(axis=1).where(education_source.notna().any(axis=1)) # identify active education column per respondent
df["education_original_code"] = np.nan # initialise raw education code column
df["education_harmonised"] = pd.Series(index=df.index, dtype="object") # initialise harmonised education column

for variable in education_variables: # populate harmonised education per country variable
    mask = df["education_source_variable"].eq(variable) # filter rows using current education variable
    codes = pd.to_numeric(df.loc[mask, variable], errors="coerce") # extract numeric response codes
    mapping = education_dictionary[education_dictionary["variable"].eq(variable)].set_index("value_code")["education_harmonised"] # build lookup series for variable
    df.loc[mask, "education_original_code"] = codes # store original numeric code
    df.loc[mask, "education_harmonised"] = codes.map(mapping) # map to harmonised education tier

df.to_csv("Task1_Cleaned_Dataset.csv", index=False) # export cleaned dataset for tasks 2 and 3
print("Task 1 completed. Cleaned dataset saved:", df.shape) # confirm export dimensions

Task 1 completed. Cleaned dataset saved: (28333, 486)


## Task 2: Validation Against Published Figures
**Performed by:** Karuna Anil Gholap (lctl0641)

### 2.1 Load Datasets and Define Pew Benchmark Toplines
Loads the cleaned and raw datasets alongside published Pew Research Center percentages for Australia, India, and South Africa on Q30 and Q31.

In [26]:
target_countries = [13, 14, 22] # define country ids for australia, india, south africa
country_names_t2 = {13: "Australia", 14: "India", 22: "South Africa"} # map country ids to display names
df = pd.read_csv("Task1_Cleaned_Dataset.csv", low_memory=False) # load task 1 cleaned dataset
raw_df = pd.read_csv("../Resources/Pew_Original_Data/Pew Research Center Global Attitudes Spring 2025 Dataset CSV.csv", low_memory=False) # load raw dataset for dk/refused check

for variable in ["country", "weight", "ai_heard", "ai_cncexc"]: # ensure key validation columns are numeric
    raw_df[variable] = pd.to_numeric(raw_df[variable], errors="coerce") # coerce non-numeric entries to nan

full_topline = { # store published pew topline percentages
    ("Q30", 13): {"A lot": 44, "A little": 53, "Nothing at all": 3, "DK/Refused": 0},
    ("Q30", 14): {"A lot": 14, "A little": 32, "Nothing at all": 35, "DK/Refused": 19},
    ("Q30", 22): {"A lot": 30, "A little": 31, "Nothing at all": 34, "DK/Refused": 6},
    ("Q31", 13): {"More excited": 13, "More concerned": 49, "Equally concerned and excited": 38, "DK/Refused": 0},
    ("Q31", 14): {"More excited": 16, "More concerned": 19, "Equally concerned and excited": 39, "DK/Refused": 26},
    ("Q31", 22): {"More excited": 18, "More concerned": 30, "Equally concerned and excited": 42, "DK/Refused": 10}
}

question_specs = { # define question variable names and response code mappings
    "Q30": {"variable": "ai_heard", "labels": {1: "A lot", 2: "A little", 3: "Nothing at all"}, "valid_codes": [1, 2, 3], "all_codes": [1, 2, 3, 8, 9]},
    "Q31": {"variable": "ai_cncexc", "labels": {1: "More excited", 2: "More concerned", 3: "Equally concerned and excited"}, "valid_codes": [1, 2, 3], "all_codes": [1, 2, 3, 8, 9]}
}

### 2.2 Compute Weighted Distributions and Verify Cleaning Consistency
Compares weighted survey estimates against Pew's published figures and verifies that Task 1 cleaning preserves valid-case distributions.

In [27]:
full_validation_rows, cleaning_check_rows = [], [] # initialise result accumulators

for question, spec in question_specs.items(): # loop over q30 and q31
    variable = spec["variable"] # get target survey column
    for country_id in target_countries: # loop over benchmark countries
        raw_country = raw_df[raw_df["country"] == country_id] # filter raw data for country
        cleaned_country = df[df["country"] == country_id] # filter cleaned data for country

        raw_full = raw_country[raw_country[variable].isin(spec["all_codes"])] # include valid and dk/refused codes
        raw_full_weight = raw_full["weight"].sum() # compute total sample weight
        for code, label in spec["labels"].items(): # compute weighted share per valid category
            calc = raw_full.loc[raw_full[variable] == code, "weight"].sum() / raw_full_weight * 100 # calculate weighted percentage
            pub = full_topline[(question, country_id)][label] # retrieve published pew percentage
            full_validation_rows.append({"Question": question, "Country": country_names_t2[country_id], "Category": label, "Raw %": round(calc, 1), "Pew %": pub, "Diff (pp)": round(calc - pub, 1)}) # record topline comparison

        dk_calc = raw_full.loc[raw_full[variable].isin([8, 9]), "weight"].sum() / raw_full_weight * 100 # calculate combined dk/refused share
        dk_pub = full_topline[(question, country_id)]["DK/Refused"] # retrieve published dk/refused share
        full_validation_rows.append({"Question": question, "Country": country_names_t2[country_id], "Category": "DK/Refused", "Raw %": round(dk_calc, 1), "Pew %": dk_pub, "Diff (pp)": round(dk_calc - dk_pub, 1)}) # record dk/refused comparison

        raw_valid = raw_country[raw_country[variable].isin(spec["valid_codes"])] # filter raw valid responses
        clean_valid = cleaned_country[cleaned_country[variable].isin(spec["valid_codes"])] # filter cleaned valid responses
        for code, label in spec["labels"].items(): # check pre- vs post-cleaning consistency
            raw_pct = raw_valid.loc[raw_valid[variable] == code, "weight"].sum() / raw_valid["weight"].sum() * 100 # compute raw valid percentage
            clean_pct = clean_valid.loc[clean_valid[variable] == code, "weight"].sum() / clean_valid["weight"].sum() * 100 # compute cleaned valid percentage
            cleaning_check_rows.append({"Question": question, "Country": country_names_t2[country_id], "Category": label, "Abs diff (pp)": abs(raw_pct - clean_pct)}) # record absolute difference

display(pd.DataFrame(full_validation_rows)) # display pew benchmark validation table
display(pd.DataFrame(cleaning_check_rows).round(4)) # display cleaning consistency table

,Question,Country,Category,Raw %,Pew %,Diff (pp)
0,Q30,Australia,A lot,43.7,44,-0.3
1,Q30,Australia,A little,53.4,53,0.4
2,Q30,Australia,Nothing at all,2.8,3,-0.2
3,Q30,Australia,DK/Refused,0.1,0,0.1
4,Q30,India,A lot,14.1,14,0.1
5,Q30,India,A little,31.6,32,-0.4
6,Q30,India,Nothing at all,35.5,35,0.5
7,Q30,India,DK/Refused,18.8,19,-0.2
8,Q30,South Africa,A lot,29.6,30,-0.4
9,Q30,South Africa,A little,31.0,31,0.0


,Question,Country,Category,Abs diff (pp)
0,Q30,Australia,A lot,0.0
1,Q30,Australia,A little,0.0
2,Q30,Australia,Nothing at all,0.0
3,Q30,India,A lot,0.0
4,Q30,India,A little,0.0
5,Q30,India,Nothing at all,0.0
6,Q30,South Africa,A lot,0.0
7,Q30,South Africa,A little,0.0
8,Q30,South Africa,Nothing at all,0.0
9,Q31,Australia,More excited,0.0


## Task 3: Generational Gap Analysis
**Performed by:** Harshini Gonela (jklt0901)

### 3.1 Construct Harmonised Age Cohorts Across All 24 Countries
Assigns respondents into `18–29`, `30–49`, and `50+` age cohorts using continuous `age` and Australia's categorical `age_australia` variable.

In [28]:
df = pd.read_csv("Task1_Cleaned_Dataset.csv", low_memory=False) # load cleaned dataset from task 1

country_names = { # map all 24 country codes to names
    2: "Canada", 3: "France", 4: "Germany", 5: "Greece", 6: "Hungary",
    7: "Italy", 8: "Netherlands", 9: "Poland", 10: "Spain", 11: "Sweden",
    12: "UK", 13: "Australia", 14: "India", 15: "Indonesia", 16: "Japan",
    17: "South Korea", 18: "Israel", 19: "Turkey", 20: "Kenya",
    21: "Nigeria", 22: "South Africa", 23: "Argentina", 24: "Brazil", 25: "Mexico"
}

df["age"] = pd.to_numeric(df["age"], errors="coerce") # coerce continuous age to numeric
df["age_australia"] = pd.to_numeric(df["age_australia"], errors="coerce") # coerce australia age bands to numeric
df["age_group"] = pd.cut(df["age"], bins=[17, 29, 49, 120], labels=["18–29", "30–49", "50+"], right=True) # bin continuous age into three cohorts

australia_mask = df["country"].eq(13) # identify australian respondents
df.loc[australia_mask, "age_group"] = df.loc[australia_mask, "age_australia"].map({1: "18–29", 2: "30–49", 3: "50+", 4: "50+"}) # map australia age bands to cohorts
df["age_group"] = pd.Categorical(df["age_group"], categories=["18–29", "30–49", "50+"], ordered=True) # set ordered categorical type
df["country_name"] = df["country"].map(country_names) # add readable country names

/var/folders/15/vnzgb5095kj14qn7m2szdjr80000gn/T/ipykernel_58524/3517116736.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["age_group"] = pd.cut(df["age"], bins=[17, 29, 49, 120], labels=["18–29", "30–49", "50+"], right=True) # bin continuous age into three cohorts
/var/folders/15/vnzgb5095kj14qn7m2szdjr80000gn/T/ipykernel_58524/3517116736.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["country_name"] = df["country"].map(country_names) # add readable country names


### 3.2 Compute Survey-Weighted Distributions by Country and Age Cohort
Calculates weighted response shares for AI awareness (`Q30`) and AI concern vs. excitement (`Q31`) across every country-cohort pair.

In [29]:
Q30_LABELS = {1: "A lot", 2: "A little", 3: "Nothing at all"} # define q30 awareness response labels
Q31_LABELS = {1: "More excited than concerned", 2: "More concerned than excited", 3: "Equally concerned and excited"} # define q31 attitude response labels

def weighted_response_distribution(data, variable, labels, question): # compute weighted percentages per country and cohort
    rows = [] # initialise output row list
    for (country_code, country_name, age_group), group in data.groupby(["country", "country_name", "age_group"], observed=True): # group by country and age cohort
        valid = group[group[variable].isin(labels.keys()) & group["weight"].notna()] # filter valid responses with weights
        denom = valid["weight"].sum() # sum weights for valid subgroup
        for code, label in labels.items(): # calculate weighted share for each option
            cat = valid[valid[variable] == code] # subset rows matching response code
            rows.append({
                "question": question, "country_code": int(country_code), "country": country_name,
                "age_group": str(age_group), "response_category": label, "n_valid": len(valid),
                "weighted_percentage": cat["weight"].sum() / denom * 100 if denom > 0 else np.nan
            }) # append subgroup result
    return pd.DataFrame(rows) # return distribution dataframe

q30_results = weighted_response_distribution(df, "ai_heard", Q30_LABELS, "Q30 – AI awareness") # compute q30 weighted distributions
q31_results = weighted_response_distribution(df, "ai_cncexc", Q31_LABELS, "Q31 – AI attitudes") # compute q31 weighted distributions

### 3.3 Calculate and Rank Cross-National Generational Gaps (`18–29` vs. `50+`)
Pivots cohort percentages to compute percentage-point differences between younger (`18–29`) and older (`50+`) adults across awareness, optimism, and concern.

In [30]:
q30_wide = q30_results.pivot_table(index=["country_code", "country", "age_group"], columns="response_category", values="weighted_percentage", aggfunc="first").reset_index() # pivot q30 to wide format
q30_wide["AI aware"] = q30_wide["A lot"] + q30_wide["A little"] # combine a lot and a little into total awareness
q31_wide = q31_results.pivot_table(index=["country_code", "country", "age_group"], columns="response_category", values="weighted_percentage", aggfunc="first").reset_index() # pivot q31 to wide format

gap_rows = [] # initialise country gap list
for country_code, country_name in sorted(country_names.items(), key=lambda x: x[1]): # iterate alphabetically across countries
    q30_c = q30_wide[q30_wide["country_code"] == country_code].set_index("age_group") # index q30 results by age group
    q31_c = q31_wide[q31_wide["country_code"] == country_code].set_index("age_group") # index q31 results by age group
    if not all(g in q30_c.index and g in q31_c.index for g in ["18–29", "50+"]): # skip country if required cohorts are missing
        continue # move to next country
    gap_rows.append({
        "country": country_name,
        "Q30 aware 18–29 (%)": q30_c.loc["18–29", "AI aware"],
        "Q30 aware 50+ (%)": q30_c.loc["50+", "AI aware"],
        "Q30 awareness gap (pp)": q30_c.loc["18–29", "AI aware"] - q30_c.loc["50+", "AI aware"],
        "Q31 excited 18–29 (%)": q31_c.loc["18–29", "More excited than concerned"],
        "Q31 excited 50+ (%)": q31_c.loc["50+", "More excited than concerned"],
        "Q31 optimism gap (pp)": q31_c.loc["18–29", "More excited than concerned"] - q31_c.loc["50+", "More excited than concerned"],
        "Q31 concerned 18–29 (%)": q31_c.loc["18–29", "More concerned than excited"],
        "Q31 concerned 50+ (%)": q31_c.loc["50+", "More concerned than excited"],
        "Q31 concern gap (pp)": q31_c.loc["18–29", "More concerned than excited"] - q31_c.loc["50+", "More concerned than excited"]
    }) # compute 18-29 minus 50+ percentage-point gaps

gap_summary = pd.DataFrame(gap_rows).sort_values("country") # create alphabetical gap summary table
display(gap_summary.round(1)) # display full country generational gap summary
display(gap_summary.assign(abs_q30=gap_summary["Q30 awareness gap (pp)"].abs()).sort_values("abs_q30", ascending=False)[["country", "Q30 awareness gap (pp)"]].round(1)) # rank countries by q30 awareness gap
display(gap_summary.assign(abs_q31=gap_summary["Q31 optimism gap (pp)"].abs()).sort_values("abs_q31", ascending=False)[["country", "Q31 optimism gap (pp)"]].round(1)) # rank countries by q31 optimism gap

,country,Q30 aware 18–29 (%),Q30 aware 50+ (%),Q30 awareness gap (pp),Q31 excited 18–29 (%),Q31 excited 50+ (%),Q31 optimism gap (pp),Q31 concerned 18–29 (%),Q31 concerned 50+ (%),Q31 concern gap (pp)
0,Argentina,82.1,61.8,20.3,21.3,10.5,10.8,25.5,55.5,-30.1
1,Australia,98.5,96.0,2.5,16.5,7.4,9.2,44.1,57.4,-13.3
2,Brazil,86.4,56.7,29.7,13.9,7.5,6.4,30.9,65.1,-34.2
3,Canada,94.2,88.6,5.6,17.1,6.0,11.1,30.1,52.1,-22.0
4,France,97.8,87.3,10.5,19.7,9.5,10.1,29.3,42.4,-13.1
5,Germany,96.3,94.7,1.7,22.0,11.4,10.6,21.1,35.5,-14.4
6,Greece,99.2,75.4,23.8,17.5,10.2,7.3,13.9,62.9,-49.0
7,Hungary,94.9,89.3,5.7,26.4,12.6,13.7,23.2,41.1,-17.9
8,India,66.2,44.7,21.5,24.3,20.6,3.7,24.9,27.5,-2.6
9,Indonesia,79.2,30.7,48.5,13.5,19.6,-6.0,31.3,37.1,-5.9


,country,Q30 awareness gap (pp)
9,Indonesia,48.5
22,Turkey,41.6
19,South Korea,36.6
14,Mexico,31.3
2,Brazil,29.7
16,Nigeria,29.3
6,Greece,23.8
13,Kenya,22.9
8,India,21.5
0,Argentina,20.3


,country,Q31 optimism gap (pp)
10,Israel,34.3
20,Spain,26.7
11,Italy,20.5
17,Poland,19.8
12,Japan,18.6
19,South Korea,16.8
15,Netherlands,16.2
7,Hungary,13.7
3,Canada,11.1
0,Argentina,10.8
